In [2]:
print('Hello')

Hello


In [3]:
!ls

LAB1.ipynb


In [4]:
!ls -all

total 4
drwxr-xr-x 1 jovyan users 4096 Mar 30 10:29 .
drwxrwxrwx 1 root   root  4096 Mar 30 10:13 ..
drwxr-xr-x 1 jovyan users 4096 Mar 30 10:13 .ipynb_checkpoints
-rw-r--r-- 1 jovyan users  747 Mar 30 10:29 LAB1.ipynb


In [5]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [6]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    print(message)
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

Writing consumer_filter.py


In [9]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value
    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Overwriting consumer_filter.py


In [10]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrichment_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

print("Uruchomiono wzbogacanie transakcji o risk_level...")

for message in consumer:
    tx = message.value
    
    # Logika risk_level
    if tx['amount'] > 3000:
        tx['risk_level'] = 'HIGH'
    elif tx['amount'] > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'
        
    print(f"WZBOGACONA: {tx['tx_id']} | Risk: {tx['risk_level']} | Kwota: {tx['amount']}")

Writing consumer_enrich.py


In [11]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='count_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Uruchomiono zliczanie per sklep...")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    
    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0) + amount
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n--- RAPORT PO {msg_count} TRANSAKCJACH ---")
        print(f"{'Sklep':<12} | {'Liczba':<7} | {'Suma':<12} | {'Średnia':<10}")
        print("-" * 50)
        for s in sorted(store_counts.keys()):
            count = store_counts[s]
            suma = total_amount[s]
            srednia = suma / count
            print(f"{s:<12} | {count:<7} | {suma:<12.2f} | {srednia:<10.2f}")

Writing consumer_count.py


In [12]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

counts = defaultdict(int)
sums = defaultdict(float)
mins = {}
maxs = {}
msg_count = 0

print("Uruchomiono statystyki per kategoria...")

for message in consumer:
    tx = message.value
    cat = tx['category']
    amt = tx['amount']
    
    counts[cat] += 1
    sums[cat] += amt
    
    if cat not in mins or amt < mins[cat]:
        mins[cat] = amt
    if cat not in maxs or amt > maxs[cat]:
        maxs[cat] = amt
        
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n>>> STATYSTYKI KATEGORII (wiadomość {msg_count}) <<<")
        print(f"{'Kategoria':<14} | {'Liczba':<6} | {'Suma':<10} | {'Min':<8} | {'Max':<8}")
        print("-" * 60)
        for c in sorted(counts.keys()):
            print(f"{c:<14} | {counts[c]:<6} | {sums[c]:<10.2f} | {mins[c]:<8.2f} | {maxs[c]:<8.2f}")

Writing consumer_stats.py


<h2>Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?</h2>

Domyślnie konsument nie odczyta żadnych historycznych zdarzeń (otrzymamy pusty wynik). Aplikacja przetwarza wyłącznie nowe rekordy, napływające w czasie rzeczywistym od momentu inicjalizacji połączenia, a nie dane historyczne - aby je odczytać należy zmienić offset konsumenta.

<h2>Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?</h2>

Oba skrypty utworzą jedną grupę konsumencką, pomiędzy którą broker spróbuje rodzielić odczyt. Kafka przydzieli całą partycję (w naszych danych mamy tylko jedną partycję) do jednego konsumenta, a drugi konsument przejdzie w tryb bezczynności (będzie węzłem zapasowym na wypadek awarii pierwszego).

<h2>Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?</h2>

Przetwarzanie stanowe: Wymaga buforowania informacji pomiędzy napływającymi zadarzeniami. Pozwala to na ciągłe śledzenie relacji między rekordami i kalkulację metryk analitycznych w czasie rzeczywistym. Wykorzystujemy je do np. agregacji danych, kalkulacji statystyk opsiwoych lub grupowania w oknach czasowych. Przetwarzanie bezstanowe polega na tym, że każdy rekord analizowany jest oddzielnie, bez budowania kontekstu historycznego. Wykorzystuje się go do zadań w locie np. filtrowania wartości odstających lub maskowania danych.


In [13]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='anomaly_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

user_history = {}

print("Uruchomiono wykrywanie anomalii...")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    
    tx_time = datetime.fromisoformat(tx['timestamp'])
    
    if user_id not in user_history:
        user_history[user_id] = []
        
    user_history[user_id].append(tx_time)
    
    recent_transactions = []
    for past_time in user_history[user_id]:
        diff_seconds = (tx_time - past_time).total_seconds()
        if diff_seconds <= 60:
            recent_transactions.append(past_time)
            
    user_history[user_id] = recent_transactions
    
    if len(user_history[user_id]) > 3:
        ilosc = len(user_history[user_id])
        print(f"ALERT FRAUD: Użytkownik {user_id} zrobił {ilosc} transakcje w ostatnie 60s!")

Writing consumer_anomaly.py
